# Hyperparameter Tuning

## Learning Objectives
1. Understand the search-space geometry and why exhaustive grid search scales poorly.
2. Implement grid search from scratch in NumPy and visualise the accuracy heatmap.
3. Compare GridSearchCV, RandomizedSearchCV, and Optuna on a real sklearn pipeline.
4. Apply Bayesian optimisation (TPE) and hyperparameter importance analysis for production model selection.

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')


## Level 1: Grid Search from Scratch (NumPy Heatmap)

In [ ]:
# Manually sweep two hyperparameters on a toy SVM and plot the accuracy heatmap.
# This illustrates the O(|C| x |gamma|) cost of grid search.

from sklearn.svm import SVC
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score

X, y = make_classification(n_samples=400, n_features=10, n_informative=5,
                           random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

C_values = [0.01, 0.1, 1.0, 10.0, 100.0]
gamma_values = [0.001, 0.01, 0.1, 1.0, 10.0]

acc_grid = np.zeros((len(C_values), len(gamma_values)))

for i, C in enumerate(C_values):
    for j, g in enumerate(gamma_values):
        clf = SVC(C=C, gamma=g, kernel='rbf')
        scores = cross_val_score(clf, X_tr, y_tr, cv=3)
        acc_grid[i, j] = scores.mean()

best_i, best_j = np.unravel_index(acc_grid.argmax(), acc_grid.shape)
print(f'Best C={C_values[best_i]}, gamma={gamma_values[best_j]}, '
      f'CV accuracy={acc_grid[best_i, best_j]:.4f}')
print(f'Grid has {len(C_values)*len(gamma_values)} cells — '
      f'grows multiplicatively with each new HP.')

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(acc_grid, aspect='auto', origin='lower', cmap='YlOrRd')
ax.set_xticks(range(len(gamma_values))); ax.set_xticklabels(gamma_values)
ax.set_yticks(range(len(C_values)));    ax.set_yticklabels(C_values)
ax.set_xlabel('gamma'); ax.set_ylabel('C')
ax.set_title('Grid Search CV Accuracy Heatmap')
plt.colorbar(im, ax=ax, label='CV Accuracy')
plt.tight_layout()
plt.savefig('/tmp/hp_grid_heatmap.png', dpi=72)
print('Heatmap saved to /tmp/hp_grid_heatmap.png')


## Level 2: GridSearchCV vs RandomizedSearchCV vs Optuna (sklearn + Optuna)

In [ ]:
# Production comparison: grid, random, and TPE-based Bayesian search.
# OOM guard: Optuna objective creates a fresh model each trial, no GPU tensors kept.

import time
from scipy.stats import loguniform, randint

X_cls, y_cls = make_classification(n_samples=800, n_features=20, n_informative=10,
                                   random_state=0)
X_tr2, X_val2, y_tr2, y_val2 = train_test_split(X_cls, y_cls, test_size=0.2,
                                                 random_state=0)

pipe = Pipeline([('scaler', StandardScaler()), ('svm', SVC(kernel='rbf'))])

# --- Grid Search ---
param_grid = {'svm__C': [0.1, 1, 10], 'svm__gamma': [0.01, 0.1, 1]}
t0 = time.time()
gs = GridSearchCV(pipe, param_grid, cv=3, n_jobs=-1)
gs.fit(X_tr2, y_tr2)
t_grid = time.time() - t0
print(f'GridSearchCV   best={gs.best_score_:.4f}  '
      f'params={gs.best_params_}  time={t_grid:.2f}s')

# --- Random Search ---
param_dist = {'svm__C': loguniform(1e-2, 1e2), 'svm__gamma': loguniform(1e-3, 1e1)}
t0 = time.time()
rs = RandomizedSearchCV(pipe, param_dist, n_iter=9, cv=3, n_jobs=-1, random_state=0)
rs.fit(X_tr2, y_tr2)
t_rand = time.time() - t0
print(f'RandomSearch   best={rs.best_score_:.4f}  '
      f'params={rs.best_params_}  time={t_rand:.2f}s')

# --- Optuna (TPE) ---
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    def objective(trial):
        C = trial.suggest_float('C', 1e-2, 1e2, log=True)
        gamma = trial.suggest_float('gamma', 1e-3, 1e1, log=True)
        clf = SVC(C=C, gamma=gamma, kernel='rbf')
        return cross_val_score(clf, X_tr2, y_tr2, cv=3).mean()

    study = optuna.create_study(direction='maximize')
    t0 = time.time()
    study.optimize(objective, n_trials=18)
    t_opt = time.time() - t0
    print(f'Optuna (TPE)   best={study.best_value:.4f}  '
          f'params={study.best_params}  time={t_opt:.2f}s')
except ImportError:
    print('optuna not installed — install with: pip install optuna')
except RuntimeError as exc:
    if 'out of memory' in str(exc).lower():
        torch.cuda.empty_cache()
        print('OOM during Optuna trial — reduce model size or disable GPU tensors')
    else:
        raise


## Real-World Example 1: Optuna with MedianPruner for Early Trial Stopping

In [ ]:
# MedianPruner stops unpromising trials early based on intermediate values.
# This is the key Optuna feature that makes it faster than random search
# on expensive objectives (deep network training).

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    from optuna.pruners import MedianPruner

    torch.manual_seed(0)
    X_t = torch.randn(600, 16, device=device)
    y_t = (X_t[:, 0] + X_t[:, 1] > 0).float().unsqueeze(1)
    ds = torch.utils.data.TensorDataset(X_t[:480], y_t[:480])
    val_X, val_y = X_t[480:], y_t[480:]

    def build_model(hidden, dropout):
        return nn.Sequential(
            nn.Linear(16, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, 1), nn.Sigmoid()
        ).to(device)

    def prune_objective(trial):
        hidden = trial.suggest_int('hidden', 16, 128)
        lr     = trial.suggest_float('lr', 1e-4, 1e-1, log=True)
        drop   = trial.suggest_float('dropout', 0.0, 0.5)
        model  = build_model(hidden, drop)
        opt    = torch.optim.Adam(model.parameters(), lr=lr)
        loss_fn = nn.BCELoss()
        loader  = torch.utils.data.DataLoader(ds, batch_size=32, shuffle=True)
        for epoch in range(20):
            model.train()
            for xb, yb in loader:
                try:
                    opt.zero_grad(); loss_fn(model(xb), yb).backward(); opt.step()
                except RuntimeError as e:
                    if 'out of memory' in str(e).lower():
                        torch.cuda.empty_cache(); break
                    raise
            model.eval()
            with torch.no_grad():
                val_acc = ((model(val_X) > 0.5).float() == val_y).float().mean().item()
            trial.report(val_acc, epoch)        # report intermediate metric
            if trial.should_prune():            # MedianPruner decision
                raise optuna.exceptions.TrialPruned()
        return val_acc

    pruner  = MedianPruner(n_startup_trials=5, n_warmup_steps=5)
    study2  = optuna.create_study(direction='maximize', pruner=pruner)
    study2.optimize(prune_objective, n_trials=20)
    pruned  = len([t for t in study2.trials if t.state == optuna.trial.TrialState.PRUNED])
    print(f'Best val accuracy: {study2.best_value:.4f}')
    print(f'Best params:       {study2.best_params}')
    print(f'Pruned trials:     {pruned} / {len(study2.trials)} '
          f'(MedianPruner eliminated {pruned/len(study2.trials)*100:.0f}%)')
except ImportError:
    print('optuna not installed. Install: pip install optuna')


## Real-World Example 2: Bayesian Optimisation with TPE Sampler

In [ ]:
# TPE (Tree-structured Parzen Estimator) models p(x|good) and p(x|bad)
# to propose candidates more likely to be in good regions.
# Key advantage over random search: uses past trials to guide future ones.

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    from optuna.samplers import TPESampler
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.metrics import f1_score

    X_gb, y_gb = make_classification(n_samples=1000, n_features=20,
                                     n_informative=12, random_state=7)
    X_gtr, X_gval, y_gtr, y_gval = train_test_split(X_gb, y_gb, test_size=0.25,
                                                     random_state=7)

    def tpe_objective(trial):
        n_estimators  = trial.suggest_int('n_estimators', 50, 300)
        max_depth     = trial.suggest_int('max_depth', 2, 8)
        learning_rate = trial.suggest_float('learning_rate', 1e-3, 0.3, log=True)
        subsample     = trial.suggest_float('subsample', 0.5, 1.0)
        clf = GradientBoostingClassifier(
            n_estimators=n_estimators, max_depth=max_depth,
            learning_rate=learning_rate, subsample=subsample, random_state=0
        )
        clf.fit(X_gtr, y_gtr)
        return f1_score(y_gval, clf.predict(X_gval))

    sampler = TPESampler(n_startup_trials=10, seed=42)
    study_tpe = optuna.create_study(direction='maximize', sampler=sampler)
    study_tpe.optimize(tpe_objective, n_trials=30)

    print(f'TPE best F1:     {study_tpe.best_value:.4f}')
    print(f'TPE best params: {study_tpe.best_params}')

    # Compare with default (random) first 10 trials as baseline
    first_10_vals = [t.value for t in study_tpe.trials[:10] if t.value is not None]
    last_10_vals  = [t.value for t in study_tpe.trials[-10:] if t.value is not None]
    if first_10_vals and last_10_vals:
        print(f'Mean F1 trials 1-10:  {np.mean(first_10_vals):.4f}')
        print(f'Mean F1 trials 21-30: {np.mean(last_10_vals):.4f}')
        print('TPE improves proposals over time (Bayesian learning effect).')
except ImportError:
    print('optuna not installed. Install: pip install optuna')


## Real-World Example 3: Hyperparameter Importance Plot

In [ ]:
# After optimising, use Optuna's FanovaImportanceEvaluator to rank
# which hyperparameters actually drove performance.
# Insight: often only 1-2 HPs matter; the rest can be fixed.

try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    from optuna.importance import get_param_importances, FanovaImportanceEvaluator

    # Reuse study_tpe from RW2 (already optimised 30 trials)
    try:
        importances = get_param_importances(
            study_tpe, evaluator=FanovaImportanceEvaluator(seed=42)
        )
        params  = list(importances.keys())
        vals    = list(importances.values())
        colors  = ['#2c7bb6' if v == max(vals) else '#abd9e9' for v in vals]

        fig, ax = plt.subplots(figsize=(6, 3))
        ax.barh(params, vals, color=colors)
        ax.set_xlabel('FANOVA Importance')
        ax.set_title('Hyperparameter Importance (GradientBoostingClassifier)')
        plt.tight_layout()
        plt.savefig('/tmp/hp_importance.png', dpi=72)
        print('Importance plot saved to /tmp/hp_importance.png')
        for p, v in importances.items():
            print(f'  {p:<20s}: {v:.4f}')
        top_hp = max(importances, key=importances.get)
        print(f'Most important HP: {top_hp} — focus tuning budget here.')
    except Exception as e:
        print(f'Importance evaluation skipped: {e}')
        # Fallback: show trial value distribution instead
        vals_all = [t.value for t in study_tpe.trials if t.value is not None]
        fig, ax = plt.subplots(figsize=(6, 3))
        ax.plot(range(len(vals_all)), vals_all)
        ax.set_xlabel('Trial'); ax.set_ylabel('F1')
        ax.set_title('Optuna Trial Progression')
        plt.tight_layout()
        plt.savefig('/tmp/hp_trials.png', dpi=72)
        print('Trial plot saved to /tmp/hp_trials.png')
except ImportError:
    print('optuna not installed. Install: pip install optuna')
    # Standalone importance proxy: correlation of each HP value with objective
    from sklearn.ensemble import GradientBoostingClassifier
    from sklearn.metrics import f1_score
    import itertools
    C_range = [0.01, 0.1, 1.0, 10.0]
    scores_hp = []
    for C_val in C_range:
        clf = SVC(C=C_val, gamma='scale')
        s = cross_val_score(clf, X_tr2, y_tr2, cv=3).mean()
        scores_hp.append(s)
    fig, ax = plt.subplots(figsize=(5, 3))
    ax.plot(C_range, scores_hp, marker='o')
    ax.set_xscale('log'); ax.set_xlabel('C'); ax.set_ylabel('CV Accuracy')
    ax.set_title('C Sensitivity (proxy for importance)')
    plt.tight_layout()
    plt.savefig('/tmp/hp_c_sensitivity.png', dpi=72)
    print('C sensitivity plot saved to /tmp/hp_c_sensitivity.png')
